compute targetibility score and return a df

In [2]:
from pathlib import Path
import mne
import numpy as np
import pandas as pd
from pcntoolkit.normative_model import NormativeModel
from pcntoolkit import NormData

import pandas as pd
import numpy as np
from pathlib import Path
import json

from ant import NFRealtime
from pathlib import Path
import argparse
import yaml
import time

/opt/homebrew/Caskroom/mambaforge/base/envs/ant-test/lib/python3.11/site-packages/ant/tools/tools.py:13: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOF


In [3]:
import py5
import numpy as np

# Create a tiny version of your data, but force it to be 
# perfectly aligned for the ARM processor
data = np.random.rand(12, 100).astype('float64')

def setup():
    py5.size(400, 400)
    # We use .tolist() to force a copy out of NumPy memory
    # into standard Python memory, which is safer for JPype
    print("Data sample:", data.tolist()[0][:5])

def draw():
    py5.background(200)
    py5.text("If you see this, py5 is working!", 50, 50)

py5.run_sketch()

Importing py5 on macOS but the necessary Jupyter macOS event loop has not been activated. I'll activate it for you, but next time, execute `%gui osx` before importing this library.


Data sample: [0.2478504578490658, 0.7173251040214403, 0.6042591868642749, 0.7956412188549453, 0.8458461208252004]


In [ ]:
export JAVA_HOME=$(/usr/libexec/java_home)
brew install ant
pip install py5 python-osc keyboard pynput

In [3]:
subjects_dir = Path("/Users/payamsadeghishabestari/temp_folder/antares_test")
models_dir = Path("/Volumes/Extreme_SSD/payam_data/Tinnorm/models/preproc_2")
subject_id = "nfru"
visit = 1
age = 21
sex = "f"
pta = 31

In [6]:
def rank_features(subject_id, subjects_dir, visit):

    subject_dir = subjects_dir / subject_id
    reliability_dir = subject_dir / "reliability_metrics"
    reliability_files = list(reliability_dir.glob(f"*_v{visit}.csv"))

    all_rankings = []
    for rel_file in reliability_files:
        df_r = pd.read_csv(rel_file)
        if not "deviation" in df_r.columns.tolist():
            continue

        for idx, row in df_r.iterrows():
            feat_name = row['feature_name']
            z_score = row['deviation']
            icc = row['icc']
            snr = row['snr']
            dynamic_range = row['dynamic_range']
            stationarity = row['stationarity']
            
            # Apply thresholds
            meets_criteria = (
                z_score >= 1.5 and  
                icc >= 0.6 and      
                snr >= 2.0 and      
                dynamic_range >= 0.3
            )
            
            # Compute composite score (weighted)
            # Normalize each metric to 0-1 range for fair weighting
            norm_z = min(z_score / 3.0, 1.0)  
            norm_icc = min(max(icc, 0), 1.0)
            norm_snr = min(snr / 10.0, 1.0)  
            norm_dr = min(dynamic_range / 2.0, 1.0)
            norm_stat = min(stationarity, 1.0)
            
            composite_score = (
                0.30 * norm_z +           # Clinical relevance
                0.25 * norm_icc +         # Reliability
                0.20 * norm_snr +         # Real-time detectability
                0.15 * norm_dr +          # Trainability
                0.10 * norm_stat          # Stability
            )
            
            all_rankings.append({
                'feature_name': feat_name,
                'feature_type': row['feature_type'],
                'z_score': z_score,
                'icc': icc,
                'snr': snr,
                'dynamic_range': dynamic_range,
                'stationarity': stationarity,
                'composite_score': composite_score,
                'meets_criteria': meets_criteria
            })

    df_ranked = pd.DataFrame(all_rankings)
    # df_ranked = df_ranked.sort_values('composite_score', ascending=False)
    # df_ranked['rank'] = range(1, len(df_ranked) + 1)

    return df_ranked

In [ ]:
subject_dir = subjects_dir / subject_id
reliability_dir = subject_dir / "reliability_metrics"
reliability_files = list(reliability_dir.glob(f"*_v{visit}.csv"))

all_rankings = []
for rel_file in reliability_files:
    df_r = pd.read_csv(rel_file)
    if not "deviation" in df_r.columns.tolist():
        continue

    for idx, row in df_r.iterrows():
        feat_name = row['feature_name']
        z_score = row['deviation']
        icc = row['icc']
        snr = row['snr']
        dynamic_range = row['dynamic_range']
        stationarity = row['stationarity']
        
        # Apply thresholds
        meets_criteria = (
            z_score >= 1.5 and  
            icc >= 0.6 and      
            snr >= 2.0 and      
            dynamic_range >= 0.3
        )
        
        # Compute composite score (weighted)
        # Normalize each metric to 0-1 range for fair weighting
        norm_z = min(z_score / 3.0, 1.0)  
        norm_icc = min(max(icc, 0), 1.0)
        norm_snr = min(snr / 10.0, 1.0)  
        norm_dr = min(dynamic_range / 2.0, 1.0)
        norm_stat = min(stationarity, 1.0)
        
        composite_score = (
            0.30 * norm_z +           # Clinical relevance
            0.25 * norm_icc +         # Reliability
            0.20 * norm_snr +         # Real-time detectability
            0.15 * norm_dr +          # Trainability
            0.10 * norm_stat          # Stability
        )
        
        all_rankings.append({
            'feature_name': feat_name,
            'feature_type': row['feature_type'],
            'z_score': z_score,
            'icc': icc,
            'snr': snr,
            'dynamic_range': dynamic_range,
            'stationarity': stationarity,
            'composite_score': composite_score,
            'meets_criteria': meets_criteria
        })

df_ranked = pd.DataFrame(all_rankings)
# df_ranked = df_ranked.sort_values('composite_score', ascending=False)
# df_ranked['rank'] = range(1, len(df_ranked) + 1)

In [12]:
df_r

,feature_name,feature_type,icc,snr,dynamic_range,stationarity,mean,std,median,p10,p90,n_epochs
0,Fp1_delta,power_sensor,-0.333333,10.994784,0.186611,0.497831,2.429496e-10,2.209681e-11,2.379748e-10,2.167112e-10,2.611200e-10,12
1,Fp2_delta,power_sensor,0.157895,8.347740,0.244909,0.466271,2.367376e-10,2.835949e-11,2.316409e-10,2.070674e-10,2.637983e-10,12
2,F3_delta,power_sensor,-1.888889,7.140452,0.302973,0.379486,2.236936e-10,3.132766e-11,2.222676e-10,1.821901e-10,2.495312e-10,12
3,F4_delta,power_sensor,0.653846,7.974166,0.297430,0.443594,2.293622e-10,2.876316e-11,2.294649e-10,1.980527e-10,2.663025e-10,12
4,C3_delta,power_sensor,0.250000,5.435313,0.500057,0.420765,2.490834e-10,4.582687e-11,2.472312e-10,1.910128e-10,3.146426e-10,12
...,...,...,...,...,...,...,...,...,...,...,...,...
615,PO8_gamma,power_sensor,0.157895,10.183612,0.249978,0.543195,1.660676e-10,1.630734e-11,1.665632e-10,1.433044e-10,1.849415e-10,12
616,Fpz_gamma,power_sensor,0.157895,14.169288,0.131483,0.418463,1.665075e-10,1.175130e-11,1.611219e-10,1.562526e-10,1.774375e-10,12
617,CPz_gamma,power_sensor,-1.181818,13.586596,0.206289,0.519289,1.585435e-10,1.166911e-11,1.588307e-10,1.425494e-10,1.753144e-10,12
618,POz_gamma,power_sensor,-1.181818,9.832929,0.263151,0.461901,1.638858e-10,1.666704e-11,1.592737e-10,1.442788e-10,1.861919e-10,12


In [11]:
reliability_files

[PosixPath('/Users/payamsadeghishabestari/temp_folder/antares_test/nfru/reliability_metrics/power_source_v1.csv'),
 PosixPath('/Users/payamsadeghishabestari/temp_folder/antares_test/nfru/reliability_metrics/conn_source_v1.csv'),
 PosixPath('/Users/payamsadeghishabestari/temp_folder/antares_test/nfru/reliability_metrics/conn_sensor_v1.csv'),
 PosixPath('/Users/payamsadeghishabestari/temp_folder/antares_test/nfru/reliability_metrics/power_sensor_v1.csv')]